# OpenCV

Dans ce tp, nous allons utiliser des outils avancés de OpenCV.

Bien sûr, OpenCV proposes tous les outils usuels de traitement de l'image, mais pour eviter la redondance avec Pillow ou Scikit-image, nous allons nous concentrer sur les modèles proposés pour la détection de visage ainsi que le traitement vidéo et la tracking d'objet.

## IMPORTANT
Avant de commencer ce tp, veillez à executer la cellule de code ci-dessous sans regarder le contenu. Celle-ci va récupérer les données et appliquer quelques transformations pour permet au TP de tourner correctement sur Colab

In [ ]:
# @title À exécuter : mise en place de l'environnement
import os
import glob
from IPython.display import HTML, clear_output
from base64 import b64encode
import cv2

if not os.path.exists("polo.mp4"):
  !pip --quiet uninstall --yes opencv-contrib-python opencv-python opencv-contrib-python-headless && pip install --quiet opencv-contrib-python
  !wget -O polo.zip https://data.votchallenge.net/sequences/acc3aeade6fb699adb7ca1c28866dd00f06d8fcd02c4cff17539a3c9d718d38976f637f4499a8763ae65f93d71fcd37494747bec7bbb8955a469b9f3fd9783aa.zip
  !unzip polo.zip -d polo
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.caffemodel.zip.001
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.caffemodel.zip.002
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.caffemodel.zip.003
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.caffemodel.zip.004
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.prototxt
  !cat goturn.caffemodel.zip.0* > ./goturn.caffemodel.zip
  !unzip goturn.caffemodel.zip
  !wget https://github.com/ultralytics/yolov5/releases/download/v6.0/yolov5s.pt
  !pip install yolov5 fastapi kaleido python-multipart uvicorn
  !wget http://vis-www.cs.umass.edu/lfw/lfw.tgz
  !tar -xf lfw.tgz
  !rm lfw.tgz

  video_name = "polo.mp4"
  images = sorted(glob.glob(f"polo/*.jpg"))
  frame = cv2.imread(images[0])
  height, width, layers = frame.shape
  video = cv2.VideoWriter(
    video_name, cv2.VideoWriter_fourcc(*"mp4v"), 10, (width, height)
  )
  for image in images:
    video.write(cv2.imread(image))
  video.release()

  !rm -rf polo
  !rm *.zip*


def show_video(video_path: str):
  # Compressed video path
  compressed_path = "displayed_video.mp4"

  os.system(f"ffmpeg -y -i {video_path}  -vcodec libx264 {compressed_path}")

  # Show video
  mp4 = open(compressed_path, "rb").read()
  data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
  display(
    HTML(
      """
  <video width=640 controls>
        <source src="%s" type="video/mp4">
  </video>
  """
      % data_url
    )
  )


clear_output()

## Imports

In [ ]:
import os
import glob
from IPython.display import HTML, clear_output
from typing import Tuple, List, Callable
from tqdm import tqdm
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from base64 import b64encode
import yolov5
from google.colab.patches import cv2_imshow

## Détection faciale

Dans cette section, vous allez mettre en place un processus de détection faciale.
À partir des données d'images dans le dossier `lfw`, récupérez une ou plusieurs images et détectez dans un rectangle le visage ainsi que les yeux, puis affichez l'image avec les détections.

La documentation OpenCV est malheureusement lacunaire, pour ne pas dire absente, en particulier pour python. Voici de quoi vous aurez besoin pour faire cet exercice par étape:
 * Chargez une image avec `cv2.imread`
 * L'outil de détection est `cv2.CascadeClassifier(model_param_path:str)`. Il prend en paramètre un chemin vers les paramètres du modèle. Vous aurez besoin de deux fichiers de paramètres. Un pour le visage et un pour les yeux que vous pourrez obtenir respectivement avec le code suivant :
   * `cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'`
   * `cv2.data.haarcascades + 'haarcascade_eye.xml'`
 * Le détecteur fonctionne sur des images en noir et blanc
 * Pour exécuter le détecteur sur une image, vous aurez besoin d'appeler la méthode `detectMultiScale`. Vous trouverez une explication détaillée des paramètres sur [`stackoverflow`](https://stackoverflow.com/questions/36218385/parameters-of-detectmultiscale-in-opencv-using-python)
 * La méthode renvoie une liste de boite qui correspondent à la détection demandée
 * Affichez le résultat de la détection sur l'image avec [`cv2.rectangle`](https://docs.opencv.org/3.1.0/d6/d6e/group__imgproc__draw.html#ga07d2f74cadcf8e305e810ce8eed13bc9) (si vous êtes perdu --> explication [`stackoverflow`](https://stackoverflow.com/questions/23720875/opencv-draw-a-rectangle-around-a-region) )
 * Vous pouvez afficher votre image avec la méthode de votre choix ou utiliser la méthode `cv2_imshow` (demandez une explication à votre gentil formateur)

_Bonne chance_

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Chargement de la cascade
face_cascade = cv2.CascadeClassifier(
  cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

# Lecture de l'image d'entrée
img = cv2.imread("lfw/Adam_Rich/Adam_Rich_0001.jpg")

# Conversion en niveaux de gris
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Détection des visages
faces = face_cascade.detectMultiScale(gray, 1.2, 5)
eyes = eye_cascade.detectMultiScale(gray, 1.2, 5)

# Dessin d'un rectangle autour des visages
for x, y, w, h in faces:
  cv2.rectangle(img, (x, y), (x + w, y + h), color=(0, 255, 0), thickness=2)
for x, y, w, h in eyes:
  cv2.rectangle(img, (x, y), (x + w, y + h), color=(255, 0, 0), thickness=2)

# Affichage de la sortie
cv2_imshow(img)

## Traitement vidéo

 ### Affichage de vidéo
 Vous allez à présent effectuer un traitement vidéo en effectuant un tracking d'un objet ou d'une personne sur une vidéo.

 Vous trouverez dans votre dossier un fichier nommé `polo.mp4`. Votre gentil formateur à prévu une fonction d'affichage pour la visualiser directement dans colab. Dans la cellule de code suivante, utiliser la fonction `show_video(video_path:str)`



In [ ]:
# Votre code ici

#### Solution

In [ ]:
show_video("polo.mp4")

### Définition du cadre de capture
Pour initialiser l'algorithme de tracking, vous aurez besoin de l'initialiser sur la première frame de la vidéo en fournissant la zone de l'objet à suivre.

Utilisez `cv2.VideoCapture` pour lire la vidéo frame par frame avec sa fonction `read`. Affichez le résultat pour vous aider à définir la zone de capture sur l'objet de votre choix.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
def plot_im_with_rect(image, rect):
  fig, ax = plt.subplots()
  ax.imshow(image)

  rect = patches.Rectangle(
    rect[:2], rect[2], rect[3], linewidth=1, edgecolor="r", facecolor="none"
  )
  ax.add_patch(rect)
  display(fig)


def plot_im_with_rects(image, rects):
  im = image.copy()
  # Draw rectangles on the original image
  for x, y, w, h in rects:
    cv2.rectangle(im, (x, y), (x + w, y + h), (0, 0, 255), 2)

  # Display the image.
  cv2_imshow(im)


HORSE_RECT = (240, 240, 120, 110)

video = cv2.VideoCapture("polo.mp4")
ret, frame = video.read()
plot_im_with_rects(frame, [HORSE_RECT])

### Tracking
Écrivez une fonction de tracking :

```python
def tracking(video_path:str, # Le chemin de la vidéo à traiter
             init_rect:Tuple[int], # Le rectange de détection initial
             model_create:Callable # Le modèle utilisé pour la capture
             )->List[np.ndarray]: # Retour la liste des frames après détection
```
Le paramètre `model_create` est une fonction permettant de créer le modèle de tracking que vous souhaitez utiliser. En voici la liste exhaustive :
* BOOSTING : `cv2.legacy.TrackerBoosting_create()`
* MIL : `cv2.TrackerMIL_create() `
* KCF : `cv2.TrackerKCF_create() `
* TLD : `cv2.legacy.TrackerTLD_create()`
* MEDIANFLOW : `cv2.legacy.TrackerMedianFlow_create()`
* GOTURN : `cv2.TrackerGOTURN_create()`
* MOSSE : `cv2.legacy.TrackerMOSSE_create()`
* CSRT : `cv2.TrackerCSRT_create()`

Choisissez celui de votre choix et produisez pour chaque frame une nouvelle frame avec la boite de tracking ([Explications des avantages et inconvénients de chaque modèle](https://learnopencv.com/object-tracking-using-opencv-cpp-python/) ).

Le tracker s'utilise de la facon suivante:
 * Créez le modèle `tracker = model_create()`
 * Initialisez le tracker avec la fonction `init` : `tracker.init(frame, init_rect)`
 * Vous pourrez maintenant appeler `tracker.update(frame)` pour effectuer la détection. La fonction renvoie deux paramètres :
  * Un booléen pour dire si la détection a fonctionné
  * La boite de détection pour l'image fournie

**ATTENTION : le tracker à besoin de boites exprimées au format xywh qui correspond aux coordonnées d'un des coins du rectangle (xy) et la taille de la boite (wh)**

*Exemple :
format xyxy (100,100,140,150) <-> format xywh (100,100,40,50)*

In [ ]:
# Votre code ici

#### Solution

In [ ]:
def tracking(
  video_path: str,
  init_rect: Tuple[int],
  model_create: Callable = cv2.TrackerGOTURN_create,
) -> List[np.ndarray]:
  frames_list = []
  video = cv2.VideoCapture("polo.mp4")
  ret, frame = video.read()
  if ret:
    tracker = model_create()
    tracker.init(frame, init_rect)
    while ret:
      ok, box = tracker.update(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
      (x, y, w, h) = (int(v) for v in box)
      if ok:
        # Tracking success
        cv2.rectangle(frame, (x, y), (x + w, y + h), color=(0, 0, 255), thickness=2)
        frames_list.append(frame)

      ret, frame = video.read()
  return frames_list

### Transformation d'une liste de frames en vidéo
Vous devriez maintenant disposer d'une liste de frames étiquetées avec les boites de tracking. Convertissez la liste des frames en une vidéo.

Pour cela, vous utiliserez `cv2.VideoWriter` qui s'utilise un peu comme `cv2.VideoCapture`, mais en écriture.
```python
cv2.VideoWriter(video_name, # Chemin de la vidéo à créer
                cv2.VideoWriter_fourcc(*'mp4v'), # Sélection du codec
                5, # Framerate
                (width,height) # Résolution
                )
```
Utilisez ensuite la fonction `write(frame)` sur chaque frame et à la toute fin du processus `video.release()` pour libérer la mémoire.

Finalement affichez votre vidéo.



*Information bonus :*

*Fourcc (Four character code) est une convention de nommage des codecs vidéo en 4 caractères. Pour des raisons de compatibilité avec google colab gardez `mp4v`, mais voici la [`liste exhaustive`](http://abcavi.kibi.ru/fourcc.php) pour votre curiosité.*

In [ ]:
# Votre code ici

#### Solution

In [ ]:
def convert_to_video(frames_list, video_name):
  height, width, _ = frames_list[0].shape
  video = cv2.VideoWriter(
    video_name, cv2.VideoWriter_fourcc(*"mp4v"), 10, (width, height)
  )

  for frame in tqdm(frames_list):
    video.write(frame)

  video.release()

In [ ]:
models = {
  "BOOSTING": cv2.legacy.TrackerBoosting_create,
  "MIL": cv2.TrackerMIL_create,
  "KCF": cv2.TrackerKCF_create,
  "TLD": cv2.legacy.TrackerTLD_create,
  "MEDIANLOW": cv2.legacy.TrackerMedianFlow_create,
  "GOTURN": cv2.TrackerGOTURN_create,
  "MOSSE": cv2.legacy.TrackerMOSSE_create,
  "CSRT": cv2.TrackerCSRT_create,
}
for model_name in models:
  print(model_name)
  frames_list = tracking("polo.mp4", HORSE_RECT, models[model_name])
  convert_to_video(frames_list, "polo_capture.mp4")
  show_video("polo_capture.mp4")

## Tracking vidéo avec YOLO
Nous allons maintenant faire du tracking avec un modèle état de l'art hors openCV appelé YOLO et présent dans la librairie yolov5 pour comparer.

YOLO fonctionne un peu différemment. Vous n'aurez pas besoin de l'initialiser et s'utilise de la façon suivante :
```python
yolo = yolov5.load('yolov5s.pt') # Chargement
...
detections = yolo(frame)
detections.xyxy # La liste des boites de détections de taille (nb_boites, 6)
```
 * La première ligne correspond au chargement des paramètres du modèle. Ils ont déjà été téléchargés pour vous dans le fichier `yolov5s.pt`
 * `detections.xyxy` contient les boites de détections. Elle contient 6 colonnes ; les 4 premières sont les coordonnées des boites au format xyxy

In [ ]:
# Votre code ici

### Solution

In [ ]:
def yolo_tracking(video_path: str) -> List[np.ndarray]:
  frames_list = []
  model = yolov5.load("yolov5s.pt")
  video = cv2.VideoCapture(video_path)
  ret, frame = video.read()
  while ret:
    detections = model(frame)

    boxes = np.array(detections.xyxy[0][:, :4]).astype(int)

    for x1, y1, x2, y2 in boxes:
      cv2.rectangle(frame, (x1, y1), (x2, y2), color=(0, 255, 0), thickness=2)
    frames_list.append(frame)
    ret, frame = video.read()
  return frames_list

In [ ]:
frames_list = yolo_tracking("polo.mp4")
convert_to_video(frames_list, "test_capture.mp4")
show_video("test_capture.mp4")